# Exemplos de Uso — carteira_auto
Guia prático de como usar o sistema.

## 1. Configuração

In [ ]:
from carteira_auto.config import settings
print(f"Portfolio: {settings.paths.PORTFOLIO_FILE}")
print(f"Lake dir: {settings.paths.LAKE_DIR}")
print(f"Yahoo timeout: {settings.yahoo.TIMEOUT}s")
print(f"Risk-free daily: {settings.portfolio.RISK_FREE_DAILY}")

## 2. Buscar Dados — Fetchers
### BCB (Banco Central)

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()
selic = bcb.get_selic(period_days=30)
print("Últimos valores da Selic:")
print(selic.tail(5))

### Yahoo Finance

In [ ]:
from carteira_auto.data.fetchers import YahooFinanceFetcher

yahoo = YahooFinanceFetcher()

# Preços múltiplos
precos = yahoo.get_multiple_prices(["PETR4", "VALE3", "ITUB4"])
for ticker, preco in precos.items():
    print(f"  {ticker}: R$ {preco:.2f}" if preco else f"  {ticker}: N/A")

### Tesouro Direto

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()
rates = tesouro.get_current_rates()
print("Taxas atuais do Tesouro Direto:")
print(rates[["tipo_titulo", "taxa_compra_manha", "taxa_venda_manha"]].head(10))

## 3. Data Lake — Persistência

In [ ]:
from carteira_auto.data.lake import DataLake
from carteira_auto.config import settings

lake = DataLake(settings.paths.LAKE_DIR)
summary = lake.summary()
print("Resumo do Data Lake:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 4. Executar Pipeline Completa

In [ ]:
from carteira_auto.core.registry import create_engine

engine = create_engine()

# Dry run (sem executar)
plan = engine.dry_run("analyze_portfolio")
print(f"Plano de execução: {plan}")

# Para executar de verdade (requer planilha Excel configurada):
# ctx = engine.run("analyze_portfolio")
# metrics = ctx["portfolio_metrics"]
# print(f"Valor total: R$ {metrics.total_value:,.2f}")

## 5. Analyzers — Uso Direto

In [ ]:
from carteira_auto.core.engine import PipelineContext
from carteira_auto.core.models import Asset, Portfolio
from carteira_auto.analyzers import PortfolioAnalyzer

# Portfolio de exemplo
portfolio = Portfolio(assets=[
    Asset(ticker="PETR4", nome="Petrobras PN", posicao_atual=5000, preco_posicao=4000, preco_atual=35, preco_medio=28, pct_meta=0.15, proventos_recebidos=200, n_cotas_atual=142),
    Asset(ticker="VALE3", nome="Vale ON", posicao_atual=3000, preco_posicao=2500, preco_atual=65, preco_medio=50, pct_meta=0.10, proventos_recebidos=100, n_cotas_atual=46),
])

ctx = PipelineContext()
ctx["portfolio"] = portfolio

analyzer = PortfolioAnalyzer()
ctx = analyzer.run(ctx)

metrics = ctx["portfolio_metrics"]
print(f"Valor total: R$ {metrics.total_value:,.2f}")
print(f"Retorno: {metrics.total_return_pct:.2%}")
print(f"Dividend Yield: {metrics.dividend_yield:.2%}")
for alloc in metrics.allocations:
    print(f"  {alloc.asset_class}: {alloc.current_pct:.1%} (meta: {alloc.target_pct:.1%}) → {alloc.action}")

## 6. Sistema de Alertas

In [ ]:
from carteira_auto.alerts.engine import AlertEngine, AlertRule

engine = AlertEngine()
engine.register_rule(AlertRule(
    name="desvio_acoes",
    condition="deviation_above",
    threshold=0.05,
    severity="warning",
    message_template="Classe {asset_class} desviou {deviation:.1%} da meta"
))

# Simular contexto com métricas
from carteira_auto.core.models import PortfolioMetrics, AllocationResult
mock_ctx = {
    "portfolio_metrics": PortfolioMetrics(
        total_value=10000, total_cost=8000, total_return=2000, total_return_pct=0.25,
        allocations=[AllocationResult(asset_class="Ações", current_pct=0.55, target_pct=0.45, deviation=0.10, action="vender")]
    )
}

alerts = engine.evaluate(mock_ctx)
for alert in alerts:
    print(f"[{alert.rule.severity}] {alert.message}")